# DAY - 18
# DATE - 06.06.2026
# U-Net for Semantic Segmentation — Satellite Application

In [30]:
import torch
import torch.nn as nn

# ==========================================
# 1. CORE REUSABLE BLOCK (Double Convolution)
# ==========================================


In [31]:
class DoubleConv(nn.Module):
   """
    Har step pe do baar convolution, batch normalization, aur ReLU apply hota hai.
    Padding=1 isliye rakha hai taaki spatial size (H x W) change na ho.
    """
   def __init__(self, in_channels, out_channels):
    super().__init__()
    self.double_conv = nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size = 3, padding = 1),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace = True),

        # Second Conv block
        nn.Conv2d(out_channels, out_channels, kernel_size = 3, padding = 1),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace = True)
    )

   def forward(self , x):
    return self.double_conv(x)


# ==========================================
# 2. MAIN U-NET ARCHITECTURE
# ==========================================


In [32]:
class MiniUNet(nn.Module):
  def __init__(self, in_channels = 13, out_classes = 4):
    super().__init__()

    # --- ENCODER (Downsampling Path) ---
    # Level 1: Input size ko process karke channels 64 karta hai

    self.inc = DoubleConv(in_channels, 64)
    self.pool1 = nn.MaxPool2d(kernel_size = 2, stride = 2)   # Size half (128 -> 64)

    # Level 2: Channels 128 karta hai
    self.down1 = DoubleConv(64, 128)
    self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2) # Size half (128 -> 64)

    # --- BOTTLENECK ---
    # U-Net ka sabse nichla hissa jahan feature extraction peak par hota hai
    self.bottleneck = DoubleConv(128, 256)

    # --- DECODER (Upsampling Path) ---
    # Level 2 Up: Size double (64 -> 128). Receives 256 -> Outputs 128 channels.
    self.up1 = nn.ConvTranspose2d(256, 128, kernel_size = 2, stride = 2)

    # DoubleConv receives (up1 + skip connection x2) = 128 + 128 = 256 channels
    self.conv_up1 = DoubleConv(256, 128)

    # Level 1 Up: Size double (128 -> 256). Receives 128 -> Outputs 64 channels.
    self.up2 = nn.ConvTranspose2d(128, 64, kernel_size = 2, stride = 2)

    # DoubleConv receives (up2 + skip connection x1) = 64 + 64 = 128 channels
    self.conv_up2 = DoubleConv(128, 64)

    # --- OUTPUT LAYER ---
    # 1x1 Conv jo final 64 features ko tumhari 4 target classes me map karega
    self.outc = nn.Conv2d(64, out_classes, kernel_size = 1)

  def forward(self, x):

    # ----------------------------------
    # ENCODER FORWARD
    # ----------------------------------
    x1 = self.inc(x)          # Shape: (Batch, 64, 256, 256)  <-- Skip Connection 1
    p1 = self.pool1(x1)       # Shape: (Batch, 64, 128, 128)

    x2 = self.down1(p1)       # Shape: (Batch, 128, 128, 128) <-- Skip Connection 2
    p2 = self.pool2(x2)       # Shape: (Batch, 128, 64, 64)

    # ----------------------------------
    # BOTTLENECK FORWARD
    # ----------------------------------

    b = self.bottleneck(p2)   ## Shape: (Batch, 256, 64, 64)

    # ----------------------------------
    # DECODER + SKIP CONNECTIONS FORWARD
    # ----------------------------------
    # Level 2 Upsampling & Merge

    up1_out = self.up1(b)     ## Shape: (Batch, 128, 128, 128)
    merge1 = torch.cat([up1_out, x2], dim=1) # Channels concatenate -> (Batch, 256, 128, 128)
    d1 = self.conv_up1(merge1)    # Shape: (Batch, 128, 128, 128)

    # Level 1 Upsampling & Merge
    up2_out = self.up2(d1)       # Shape: (Batch, 64, 256, 256)
    merge2 = torch.cat([up2_out, x1], dim=1) # Channels concatenate -> (Batch, 128, 256, 256)
    d2 = self.conv_up2(merge2)    # Shape

    # ----------------------------------
    # FINAL OUTPUT
    # ----------------------------------
    logits = self.outc(d2)      # Shape: (Batch, 4, 256, 256)
    return logits

In [33]:
if __name__ == "__main__":
  # Model initialization
  model = MiniUNet(in_channels=13, out_classes=4)

  #  Fake Batch Simulation: [Batch size = 1, Channels = 13, Height = 256, Width = 256]
  dummy_input = torch.randn(1, 13, 256, 256)
  output = model(dummy_input)

  # Summary Print
  print("\n" + "="*30)
  print("       🚀 MINI U-NET SUMMARY       ")
  print("="*30)
  print(f"📥 Input Shape  : {list(dummy_input.shape)}  (1 Satellite Image, 13 Bands)")
  print(f"📤 Output Shape : {list(output.shape)}  (1 Predicted Mask, 4 Classes)")
  print("="*30)
  print("✅ Model Architecture Verified Successfully!\n")


       🚀 MINI U-NET SUMMARY       
📥 Input Shape  : [1, 13, 256, 256]  (1 Satellite Image, 13 Bands)
📤 Output Shape : [1, 4, 256, 256]  (1 Predicted Mask, 4 Classes)
✅ Model Architecture Verified Successfully!

